# Challenge 03: API May Reject Frequent Requests

**PPT problem:** API may reject frequent requests  
**Technique:** Rate limiting / Retry-After

A provider uses rate limits to protect service availability. The correct client behavior is not to retry immediately. Read the 429 response and Retry-After header, then wait before continuing.


## Setup

Make sure the mock serving API is running:

```bash
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --port 8011
```

If you use a different local port, update `BASE_URL` in the code cell.


## Run the Broken Program First

Do not fix the code before running it. Run it once and observe the failure signal.

Look for:

- On which request does the script fail?
- What HTTP status code appears?
- Is there a Retry-After header?



In [1]:
"""
Challenge 03: Rate limit.

The API is healthy, but this client calls it too quickly. The provider returns
429 Too Many Requests. Fix the client so it respects the response.
"""

import time

import pandas as pd
import requests


BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = f"group_03_rate_limit_{time.time_ns()}"


def main() -> None:
    all_records = []

    for page in range(1, 7):
        response = requests.get(
            f"{BASE_URL}/debug-lab/03-rate-limit/weather/records",
            params={
                "page": page,
                "page_size": 5,
                "client_id": CLIENT_ID,
            },
            timeout=10,
        )
        response.raise_for_status()
        all_records.extend(response.json()["data"])

    dataframe = pd.DataFrame(all_records)
    print("Weather rows collected:", len(dataframe))
    print(dataframe[["area", "forecast"]].head())


if __name__ == "__main__":
    main()


HTTPError: 429 Client Error: Too Many Requests for url: http://127.0.0.1:8011/debug-lab/03-rate-limit/weather/records?page=4&page_size=5&client_id=group_03_rate_limit_1782180551404086000

## Diagnose

Write short answers before changing the code.

1. What failed: HTTP request, Python processing, or data quality check?
2. What exact error/status/message did you see?
3. Which response header or body field should guide the wait time?
4. Which technique from the PPT table should fix it?


In [ ]:
# Notes
# 1. http request error
# 2. 429 too many requests
# 3. header retry-after - retry after tells how long my application need to wait before try again
# 4.

## Where to Look for the Fix

Use the API docs, the observed failure, and the clues below to decide what to change.

Primary place to check: `http://127.0.0.1:8011/docs`, then find `/debug-lab/03-rate-limit/weather/records`.

Use these clues:

1. Open `/docs` and inspect the endpoint behavior.
2. When you get a 429 response, check the response headers and body before retrying.
3. Look for a wait time such as `Retry-After`, then slow the client down.

After that, use the success condition below to check whether your repaired code is good enough.


## Repair Workspace

Copy the broken code here and edit it until it works.

Success condition: The notebook should collect weather pages without being blocked by rate limits.


In [3]:
# Paste the broken code here, then repair it.
# Start by checking /docs for the endpoint contract and rereading the error output above.
# Stop when the success condition in the previous markdown cell is satisfied.
"""
Challenge 03: Rate limit.

The API is healthy, but this client calls it too quickly. The provider returns
429 Too Many Requests. Fix the client so it respects the response.
"""

import time

import pandas as pd
import requests

TEMPORARY_STATUS_CODES = {429, 500, 502, 503, 504}
BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = f"group_03_rate_limit_{time.time_ns()}"

    
def get_with_retry(url, params=None, max_attempts=5, timeout_sec = 5, base_sleep=1):
    # Try the same request multiple times before giving up.
    for attempt in range(1, max_attempts + 1):
        print(f"Attempt {attempt}/{max_attempts}: sending request to the API")
        try:
            response = requests.get(url, params=params, timeout=timeout_sec)
            print(f"Received status code: {response.status_code}")

            if response.status_code == 429:
                retry_after = int(response.headers.get("Retry-After", base_sleep))
                print(f"Request was rate limited. Waiting {retry_after}s before retrying")
                time.sleep(retry_after)
                continue

            if response.status_code in TEMPORARY_STATUS_CODES:
                wait_seconds = base_sleep * (2 ** (attempt - 1))
                print(f"Temporary API error: HTTP {response.status_code}")
                print(f"Retrying after {wait_seconds}s")
                time.sleep(wait_seconds)
                continue

            response.raise_for_status()
            print("Request successful")
            return response
        except requests.Timeout:
            wait_seconds = base_sleep * (2 ** (attempt - 1))
            print("Request timed out")
            print(f"Retrying after {wait_seconds}s")
            time.sleep(wait_seconds)

    print("No retry attempts left")
    raise RuntimeError(f"Request failed after {max_attempts} attempts: {url}")


def main() -> None:
    all_records = []

    for page in range(1, 7):
        response = get_with_retry(
            f"{BASE_URL}/debug-lab/03-rate-limit/weather/records",
            params={
                "page": page,
                "page_size": 5,
                "client_id": CLIENT_ID,
            },
            timeout_sec=10,
        )
        all_records.extend(response.json()["data"])

    dataframe = pd.DataFrame(all_records)
    print("Weather rows collected:", len(dataframe))
    print(dataframe[["area", "forecast"]].head())


if __name__ == "__main__":
    main()




Attempt 1/5: sending request to the API
Received status code: 200
Request successful
Attempt 1/5: sending request to the API
Received status code: 200
Request successful
Attempt 1/5: sending request to the API
Received status code: 200
Request successful
Attempt 1/5: sending request to the API
Received status code: 429
Request was rate limited. Waiting 5s before retrying
Attempt 2/5: sending request to the API
Received status code: 200
Request successful
Attempt 1/5: sending request to the API
Received status code: 200
Request successful
Attempt 1/5: sending request to the API
Received status code: 200
Request successful
Weather rows collected: 30
          area             forecast
0   Ang Mo Kio  Partly Cloudy (Day)
1        Bedok  Partly Cloudy (Day)
2       Bishan  Partly Cloudy (Day)
3     Boon Lay  Partly Cloudy (Day)
4  Bukit Batok  Partly Cloudy (Day)
